<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/SectorLeadershipModern.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade yfinance

In [1]:
import seaborn as sns
import yfinance as yf
print(yf.__version__)
import pandas as pd
#import pandas_ta as ta
import numpy as np
from datetime import datetime
import time
import matplotlib.pyplot as plt


print("Libraries Installed!")

0.2.66
Libraries Installed!


## Define Sector and Benchmark

In [7]:
tickers = [
    'XLK','XLF','XLV','XLI','XLY', 'XBI', 'GBTC', 'NVDA',
    'XLP','XLE','XLU','XLB','XLRE','SOXX', 'GBTC', 'GLD','SLV',
    'XLC','SPY'
]

prices = yf.download(
    tickers,
    start="2020-01-01",
    auto_adjust=True
)['Close']

weekly = prices.resample('W-FRI').last()

#current_week_end = weekly.index[-1]
#previous_week_end = weekly.index[-2]
# Optional manual override
current_week_end = pd.Timestamp('2026-06-13')
previous_week_end = pd.Timestamp('2026-06-20')

# =============================
# 2. SCORE FUNCTION (FIXED RS + ACCEL)
# =============================
def calculate_score(data, as_of):

    df = data.loc[:as_of]

    sector = df.drop(columns="SPY")

    # -----------------------------
    # Relative Strength
    # -----------------------------
    rs = sector.div(df["SPY"], axis=0)

    # -----------------------------
    # RS Momentum (4-week change)
    # -----------------------------
    rs_momentum = rs.pct_change(4)

    # -----------------------------
    # RS Acceleration (change in momentum)
    # -----------------------------
    rs_accel = rs_momentum.diff(1)

    # -----------------------------
    # RS Slope (trend of RS)
    # -----------------------------
    rs_slope = (
        rs.rolling(4).mean().iloc[-1] -
        rs.rolling(4).mean().iloc[-4]
    )

    # -----------------------------
    # Latest snapshot
    # -----------------------------
    score = pd.DataFrame({
        "RS_Strength": rs.iloc[-1],
        "RS_Momentum": rs_momentum.iloc[-1],
        "RS_Accel": rs_accel.iloc[-1],
        "RS_Slope": rs_slope
    })

    # Composite rank score
    score["Total"] = score.rank(ascending=False).mean(axis=1)

    return score


# =============================
# 3. CURRENT / PREVIOUS SCORES
# =============================
current_score = calculate_score(weekly, current_week_end)
previous_score = calculate_score(weekly, previous_week_end)

# =============================
# 4. RANKS
# =============================
curr_rank = current_score["Total"].rank()
prev_rank = previous_score["Total"].rank()

# =============================
# 5. IMPACT (NOW USING ACCELERATION)
# =============================
impact = (prev_rank - curr_rank) * current_score["RS_Accel"].abs()

# =============================
# 6. ROTATION TABLE
# =============================
df = pd.DataFrame({
    "Curr Rank": curr_rank,
    "Prev Rank": prev_rank,
    "Rank Change": prev_rank - curr_rank,
    "RS_Strength": current_score["RS_Strength"],
    "RS_Momentum": current_score["RS_Momentum"],
    "RS_Accel": current_score["RS_Accel"],
    "Impact": impact
})

df = df.sort_values("Impact", ascending=False)

# =============================
# 7. TOP 5 ROTATION RATE
# =============================
N = 5

top_now = set(current_score.sort_values("Total").head(N).index)
top_prev = set(previous_score.sort_values("Total").head(N).index)

rotating_in = top_now - top_prev
rotating_out = top_prev - top_now

rotation_rate = len(rotating_in) / N

# =============================
# 8. REGIME CLASSIFICATION
# =============================
if rotation_rate == 0:
    regime = "Stable Leadership"
elif rotation_rate <= 0.2:
    regime = "Low Rotation"
elif rotation_rate <= 0.4:
    regime = "Mild Rotation"
elif rotation_rate <= 0.6:
    regime = "Moderate Rotation"
else:
    regime = "High Rotation"

# =============================
# 9. FLOW LABELS
# =============================
def flow(x):
    if x >= 3:
        return "Strong Inflow 🚀"
    elif x >= 1:
        return "Mild Inflow 📈"
    elif x <= -3:
        return "Strong Outflow 🔻"
    elif x <= -1:
        return "Mild Outflow 📉"
    else:
        return "Neutral"

df["Flow"] = df["Rank Change"].apply(flow)

# =============================
# 10. OUTPUT
# =============================
print("\n==============================")
print("SECTOR ROTATION ENGINE (v2)")
print("==============================")

print(f"Current Week  : {current_week_end.date()}")
print(f"Previous Week : {previous_week_end.date()}")
print(f"Rotation Rate : {rotation_rate:.0%}")
print(f"Regime        : {regime}")

print("\nRotating IN:")
print(rotating_in if rotating_in else "None")

print("\nRotating OUT:")
print(rotating_out if rotating_out else "None")



[*********************100%***********************]  18 of 18 completed


SECTOR ROTATION ENGINE (v2)
Current Week  : 2026-06-13
Previous Week : 2026-06-20
Rotation Rate : 40%
Regime        : Mild Rotation

Rotating IN:
{'XLB', 'XLV'}

Rotating OUT:
{'NVDA', 'XLI'}


In [9]:
# =============================
# SECTOR SIGNAL CLASSIFICATION
# =============================

def sector_signal(row, rank_series):

    rank = rank_series[row.name]
    accel = row["RS_Accel"]
    rank_change = row["Rank Change"]

    # Top, mid, bottom segmentation
    if rank <= 5:
        tier = "top"
    elif rank <= 10:
        tier = "mid"
    else:
        tier = "weak"

    # -----------------------------
    # BUY CONDITIONS
    # -----------------------------
    if tier == "top" and accel > 0 and rank_change > 0:
        return "BUY 🟢"

    if tier == "top" and accel > 0:
        return "BUY 🟢 (early)"

    # -----------------------------
    # WATCH CONDITIONS
    # -----------------------------
    if tier == "mid" and accel >= 0:
        return "WATCH 🟡 (improving)"

    if tier == "top" and accel <= 0:
        return "WATCH 🟡 (late cycle)"

    if tier == "mid" and rank_change > 0:
        return "WATCH 🟡 (building)"

    # -----------------------------
    # AVOID CONDITIONS
    # -----------------------------
    if tier == "weak" and accel < 0:
        return "AVOID 🔴"

    if rank_change < 0 and accel < 0:
        return "AVOID 🔴 (distribution)"

    return "WATCH 🟡"

In [10]:
df["Signal"] = df.apply(
    sector_signal,
    axis=1,
    rank_series=curr_rank
)

final_view = df.sort_values(
    ["Signal", "Impact"],
    ascending=[True, False]
)

print("\n==============================")
print("SECTOR SIGNAL DASHBOARD")
print("==============================")

final_view


SECTOR SIGNAL DASHBOARD


,Curr Rank,Prev Rank,Rank Change,RS_Strength,RS_Momentum,RS_Accel,Impact,Flow,Signal
Ticker,,,,,,,,,
XLE,16.5,17.0,0.5,0.077230,-0.035164,-0.070631,0.035315,Neutral,AVOID 🔴
NVDA,14.5,5.5,-9.0,0.277342,-0.091449,-0.045718,-0.411460,Strong Outflow 🔻,AVOID 🔴
XLB,5.0,8.0,3.0,0.070266,0.033768,0.052283,0.156848,Strong Inflow 🚀,BUY 🟢
SOXX,1.0,1.0,0.0,0.805531,0.168442,0.130923,0.000000,Neutral,BUY 🟢 (early)
XBI,3.0,3.0,0.0,0.180658,0.020159,0.064906,0.000000,Neutral,BUY 🟢 (early)
XLK,2.0,2.0,0.0,0.249485,0.044804,0.017474,0.000000,Neutral,BUY 🟢 (early)
SLV,13.0,16.0,3.0,0.082842,-0.115342,0.041269,0.123808,Strong Inflow 🚀,WATCH 🟡
XLU,11.0,13.0,2.0,0.059806,0.011514,0.019693,0.039387,Mild Inflow 📈,WATCH 🟡
XLC,14.5,11.5,-3.0,0.150519,-0.041509,0.003466,-0.010399,Strong Outflow 🔻,WATCH 🟡
